## pipeline step 5: OCR

tasks:
* perform OCR on all images of one directory
* collect results in a pandas dataframe and store on disk

In [1]:
import cv2
import pandas as pd
import warnings
from pathlib import Path
import easyocr
import Levenshtein
import datetime

* settings for EasyOCR:

In [2]:
# Silence non-critical EasyOCR warnings
warnings.filterwarnings("ignore", message=r".*pin_memory.*no accelerator.*", category=UserWarning)

# EasyOCR parameters
LANGS = ["en"]

OCR_PARAMS = dict(
    detail=1,
    paragraph=False,
    allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz",
    text_threshold=0.5,
    low_text=0.3,
    link_threshold=0.3,
    mag_ratio=2.0,
)

# Init reader
reader = easyocr.Reader(LANGS, gpu=False)


Using CPU. Note: This module is much faster with a GPU.


* helper functions:

In [3]:
# Helper functions
def preprocess(img):
    """
    Preprocess each image to help easyOCR
    Simple binarize + slight dilation (helps dotty letters)
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    th = cv2.dilate(th, kernel, iterations=1)
    return cv2.cvtColor(th, cv2.COLOR_GRAY2BGR)

def normalize(s):
    """
    Function to remove all non-alphanumeric characters, and convert characters to lowercase
    """
    #return re.sub(r"[^a-zA-Z0-9]", "", s.lower()) if s else ""
    return s.lower()


process one directory:
* create output file for the dataframe, its name contains the full directory path and timestamp
* get all images of the directory
* get levenshtein distance for each image
* get ground truth of each image (available in the filename)

In [4]:
def process_dir(image_dir):
    print('***', image_dir)
    rows = []

    # make filename from a full directory path
    csv_output = image_dir.replace('.', '').replace('\\', '_').replace('/', '_')
    #print(filename_output)
    csv_output += datetime.datetime.now().strftime("_%Y%m%d_%H%M%S")
    csv_output += '.csv'

    # Get paths of all images
    image_files = Path(image_dir).glob("*.png")
    image_files = list(image_files)
    print([i.stem for i in image_files[:5]+image_files[-5:]])

    # Loop over all images to predict the words
    for image_filename in image_files:

        # general:
        gt = image_filename.stem.split('_')[0]
        source = '_'.join(image_filename.stem.split('_')[:2]) # only take name + id
        # only for edge_output:
        gt = image_filename.stem.split('_')[1]
        source = '_'.join(image_filename.stem.split('_')[1:3]) # only take name + id

        img = cv2.imread(str(image_filename))
        img = preprocess(img)

        ocr_out = reader.readtext(img, **OCR_PARAMS)

        pred = " ".join([x[1] for x in ocr_out])

        conf = 0.0
        total_len=0
        for part_box, part_pred, part_conf in ocr_out:
            conf += len(part_pred)*part_conf
            total_len += len(part_pred)
        if total_len>0:
            conf /= total_len

        pred_n = normalize(pred)
        gt_n = normalize(gt)

        exact = pred == gt
        exact_n = pred_n == gt_n
        le_gt = len(gt)
        le_pred = len(pred)
        dist = Levenshtein.distance(gt, pred)
        dist_n = Levenshtein.distance(gt_n, pred_n)
        error_rate = min(dist / le_gt, 1.0)
        error_rate_n = min(dist_n / le_gt, 1.0)
        match = 1.0 - error_rate
        match_n = 1.0 - error_rate_n

        print(image_filename.name, "'"+pred+"'",gt, conf, match)

        rows.append({
            "file_name": image_filename.name,
            "source": source,
            "ground_truth": gt,
            "prediction": pred,
            "exact_match": exact,
            "exact_match_n": exact_n,
            "distance": dist,
            "distance_norm": dist_n,
            "error_rate": error_rate,
            "error_rate_norm": error_rate_n,
            "match": match,
            "match_norm": match_n,
            "confidence": conf,
        })
    df = pd.DataFrame(rows)

    # Save results to csv
    df.to_csv(csv_output, index=False)

In [6]:
# set directories to process:
image_dirs = [r"..\data\DATA_FINAL\test_input",
              r"..\data\DATA_FINAL\test_target",
]

for i in image_dirs:
    process_dir(i)

*** ..\data\edge_output_1
['edge_albeit_007550_input', 'edge_albeit_007551_input', 'edge_assess_007572_input', 'edge_assess_007573_input', 'edge_audit_007538_input', 'edge_unit_007475_input', 'edge_walker_007570_input', 'edge_walker_007571_input', 'edge_yet_007440_input', 'edge_yet_007441_input']
edge_albeit_007550_input.png 'K BKT' albeit 0.05769373529787468 0.0
edge_albeit_007551_input.png 'Le Xx' albeit 0.13480543701215886 0.0
edge_assess_007572_input.png '' assess 0.0 0.0
edge_assess_007573_input.png '' assess 0.0 0.0
edge_audit_007538_input.png 'Zp' audit 0.3135602543025773 0.0
edge_audit_007539_input.png 'Ti' audit 0.4855585578339536 0.19999999999999996
edge_barber_007586_input.png '' barber 0.0 0.0
edge_barber_007587_input.png 'S sEa' barber 0.10707761237515714 0.0
edge_bean_007486_input.png 'EV' bean 0.0491796749878444 0.0
edge_bean_007487_input.png '' bean 0.0 0.0
edge_beast_007520_input.png 'F' beast 0.4015167098198482 0.0
edge_beast_007521_input.png 'REIs' beast 0.3631683588